# 🔬 Pipeline Atômico com ColumnTransformer & Protocolo Anti-Leakage
**Projeto:** Sanidade-Vegetal (SugarVision)  
**Sprint:** 2 — Framework SEMMA (Fase: Modify)  
**Responsável:** Elisa (`EA`) — Engenharia de Machine Learning & MLOps  
**Dataset Base:** `data/processed/abt_sanidade_vegetal.csv` (6.571 instâncias)  

---

## 📌 Objetivo da Entrega

Desenvolver e validar o **fluxo unificado de pré-processamento no Scikit-Learn** utilizando `Pipeline` e `ColumnTransformer`, encapsulando imputação por mediana, padronização estatística z-score e codificação dummy (One-Hot) com proteção estrita contra vazamento de dados (*Data Leakage*), garantindo conformidade com a **Regra de Ouro do Fit** e compatibilidade com MLOps.

### ✅ Cobertura do Checklist da Sprint 2:
1. **Mapear as listas de colunas** conforme o tratamento requerido (numéricas, discretizadas e categóricas).
2. **Criar o sub-pipeline numérico** com `SimpleImputer(strategy='median')` e `StandardScaler()`.
3. **Criar o sub-pipeline categórico** com `OneHotEncoder(sparse_output=False, handle_unknown='ignore')`.
4. **Integrar todos os transformadores** em um único objeto `ColumnTransformer`.
5. **Validar a "Regra de Ouro do Fit":** assegurar que o `.fit()` ocorra apenas no treino e o `.transform()` seja replicado nas demais partições.
6. **Validar a serialização preliminar** do pipeline para checar compatibilidade futura com MLOps.

In [1]:
# 1. Configuração do ambiente e importação das dependências
import os
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

# Configurações de exibição do pandas e sklearn
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
sklearn.set_config(display='diagram')  # Renderização visual interativa do pipeline

print(f"Scikit-Learn version : {sklearn.__version__}")
print(f"Joblib version       : {joblib.__version__}")
print(f"Pandas version       : {pd.__version__}")

Scikit-Learn version : 1.9.1
Joblib version       : 1.6.0
Pandas version       : 3.0.6


---
## 1. Carga e Inspeção Estrutural da Tabela Analítica Base (ABT)

Carregamos o dataset consolidado `data/processed/abt_sanidade_vegetal.csv`, contendo os metadados de imagem, variáveis de engenharia espectral/textura Haralick e variáveis discretizadas via binning.

In [2]:
# Localização do dataset no projeto
base_dir = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
abt_path = base_dir / "data" / "processed" / "abt_sanidade_vegetal.csv"

df = pd.read_csv(abt_path)
print(f"Dimensão da ABT: {df.shape[0]} registros x {df.shape[1]} colunas.")

# Distribuição de partições estritas
split_dist = df['split_partition'].value_counts()
print("\nDistribuição das partições experimentais:")
for split, count in split_dist.items():
    print(f"  • {split:6s}: {count:5d} amostras ({count/len(df)*100:5.2f}%)")

Dimensão da ABT: 6571 registros x 35 colunas.

Distribuição das partições experimentais:
  • train :  5574 amostras (84.83%)
  • valid :   617 amostras ( 9.39%)
  • test  :   380 amostras ( 5.78%)


---
## 2. Taxonomia e Mapeamento de Colunas (Checklist 1)

Mapeamos explicitamente os subconjuntos de variáveis conforme os requisitos do cartão:
- **Numéricas Contínuas (17 features):** Índices cromáticos, texturas Haralick GLCM e metadados de tamanho e resolução física.
- **Discretizadas (4 features):** Intervalos de binning de `size_kb_bin`, `laplacian_var_bin`, `resolution_bin`, `aspect_ratio_bin`.
- **Categóricas de Metadados (2 features):** `dataset_source` e `extension`.
- **Categóricas Totais:** Unificação das nominais e discretizadas para codificação One-Hot.
- **Metadados Descartados:** Identificadores, nomes de arquivos e dimensões redundantes (`remainder='drop'`).
- **Alvos Supervisionados:** `class_label`, `target_binary`, `target_multiclass`.

In [3]:
# 1. Features contínuas (Cromáticas, Textura Haralick e Metadados Físicos de Imagem)
NUMERICAL_FEATURES = [
    'mean_hue', 'std_saturation', 'exg_index', 'exr_index', 'rg_ratio',
    'indice_clorose_necrose', 'hue_dispersion', 'glcm_contrast',
    'glcm_homogeneity', 'glcm_dissimilarity', 'glcm_energy',
    'indice_rugosidade_pustula', 'laplacian_var', 'size_kb',
    'total_pixels', 'resolution_mp', 'aspect_ratio'
]

# 2. Features discretizadas em faixas (bins estatísticos)
DISCRETIZED_FEATURES = [
    'size_kb_bin', 'laplacian_var_bin', 'resolution_bin', 'aspect_ratio_bin'
]

# 3. Features categóricas de metadados
CATEGORICAL_NOMINAL_FEATURES = ['dataset_source', 'extension']

# Unificação para o OneHotEncoder
ALL_CATEGORICAL_FEATURES = CATEGORICAL_NOMINAL_FEATURES + DISCRETIZED_FEATURES

# 4. Colunas de metadados/identificação excluídas do vetor preditivo
METADATA_DROPPED = [
    'sample_id', 'filepath', 'filename', 'relative_path',
    'width', 'height', 'channels', 'color_mode'
]

# 5. Targets supervisionados
TARGET_COLS = ['class_label', 'target_binary', 'target_multiclass']

summary_df = pd.DataFrame([
    {"Grupo": "Numéricas Contínuas", "Quantidade": len(NUMERICAL_FEATURES), "Estratégia": "SimpleImputer(median) + StandardScaler()"},
    {"Grupo": "Categóricas Nominais", "Quantidade": len(CATEGORICAL_NOMINAL_FEATURES), "Estratégia": "SimpleImputer(most_frequent) + OneHotEncoder()"},
    {"Grupo": "Discretizadas (Bins)", "Quantidade": len(DISCRETIZED_FEATURES), "Estratégia": "OneHotEncoder(sparse_output=False, handle_unknown='ignore')"},
    {"Grupo": "Metadados Isolados", "Quantidade": len(METADATA_DROPPED), "Estratégia": "remainder='drop'"},
    {"Grupo": "Targets Supervisionados", "Quantidade": len(TARGET_COLS), "Estratégia": "Preservação Externa (Y)"}
])
display(summary_df)

,Grupo,Quantidade,Estratégia
0,Numéricas Contínuas,17,SimpleImputer(median) + StandardScaler()
1,Categóricas Nominais,2,SimpleImputer(most_frequent) + OneHotEncoder()
2,Discretizadas (Bins),4,"OneHotEncoder(sparse_output=False, handle_unkn..."
3,Metadados Isolados,8,remainder='drop'
4,Targets Supervisionados,3,Preservação Externa (Y)


---
## 3. Construção do Pipeline Atômico com ColumnTransformer (Checklists 2, 3 e 4)

Encapsulamos o fluxo com:
- **Sub-pipeline numérico:** Imputação por mediana (resistente a assimetrias na iluminação foliar) seguida de normalização z-score.
- **Sub-pipeline categórico:** Imputação de moda e `OneHotEncoder(sparse_output=False, handle_unknown='ignore')` para prevenir falhas em categorias inéditas de produção.
- **ColumnTransformer:** Agrupamento atômico com `remainder='drop'` e saída formatada para DataFrame pandas nativo (`set_output(transform='pandas')`).

In [4]:
# Sub-pipeline Numérico
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Sub-pipeline Categórico (Nominais + Discretizadas)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# ColumnTransformer Unificado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERICAL_FEATURES),
        ('cat', categorical_transformer, ALL_CATEGORICAL_FEATURES)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# Configurar saída pandas nativa para auditoria e rastreabilidade imediata
preprocessor.set_output(transform='pandas')

# Renderização do diagrama interativo
display(preprocessor)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The

---
## 4. Validação da "Regra de Ouro do Fit" (Anti-Leakage) (Checklist 5)

> **Regra de Ouro:** O aprendizado de parâmetros estatísticos ($\mu, \sigma$, medianas, categorias válidas) deve ocorrer **estritamente sobre o conjunto de treino** via `.fit()` ou `.fit_transform()`. As partições de validação e teste recebem exclusivamente `.transform()`, garantindo que nenhuma contaminação de dados (*information leakage*) ocorra.

In [5]:
# 1. Divisão estrita conforme split_partition
df_train = df[df['split_partition'] == 'train'].copy()
df_valid = df[df['split_partition'] == 'valid'].copy()
df_test  = df[df['split_partition'] == 'test'].copy()

print(f"• Amostras de Treino:     {len(df_train)}")
print(f"• Amostras de Validação:  {len(df_valid)}")
print(f"• Amostras de Teste:      {len(df_test)}")

# 2. Ajuste EXCLUSIVO no conjunto de Treino
X_train_trans = preprocessor.fit_transform(df_train)

# 3. Replicação com .transform() em Validação e Teste
X_valid_trans = preprocessor.transform(df_valid)
X_test_trans  = preprocessor.transform(df_test)

print(f"\nShape pós-processamento:")
print(f"  - X_train_trans : {X_train_trans.shape}")
print(f"  - X_valid_trans : {X_valid_trans.shape}")
print(f"  - X_test_trans  : {X_test_trans.shape}")

# 4. Verificação de ausência de NaNs
assert X_train_trans.isna().sum().sum() == 0, "Existem NaNs no Treino!"
assert X_valid_trans.isna().sum().sum() == 0, "Existem NaNs na Validação!"
assert X_test_trans.isna().sum().sum() == 0, "Existem NaNs no Teste!"
print("\n✓ [VALIDADO] Ausência total de NaNs em todas as partições.")

# 5. Auditoria Matemática dos Parâmetros Aprendidos
imputer_step = preprocessor.named_transformers_['num'].named_steps['imputer']
scaler_step  = preprocessor.named_transformers_['num'].named_steps['scaler']

# Conferir se a mediana bate com o Treino
expected_train_medians = df_train[NUMERICAL_FEATURES].median().values
np.testing.assert_allclose(imputer_step.statistics_, expected_train_medians, rtol=1e-5)
print("✓ [VALIDADO] Imputer medians correspondem estritamente ao Treino (Sem Leakage).")

# Conferir se as médias numéricas no Treino são 0.0
num_cols_trans = [col for col in X_train_trans.columns if col.startswith('num__')]
train_means = X_train_trans[num_cols_trans].mean()
assert np.all(np.abs(train_means) < 1e-6), "Médias de treino não padronizadas para zero!"
print("✓ [VALIDADO] Média das variáveis numéricas de Treino centrada em 0.0.")

# Comparação com o dataset total para provar o isolamento
full_means = df[NUMERICAL_FEATURES].mean().values
mean_diff = np.abs(scaler_step.mean_ - full_means)
print(f"✓ [VALIDADO] Desvio médio entre os parâmetros do Treino vs População Total: {mean_diff.mean():.6f}")

• Amostras de Treino:     5574
• Amostras de Validação:  617
• Amostras de Teste:      380



Shape pós-processamento:
  - X_train_trans : (5574, 35)
  - X_valid_trans : (617, 35)
  - X_test_trans  : (380, 35)



✓ [VALIDADO] Ausência total de NaNs em todas as partições.
✓ [VALIDADO] Imputer medians correspondem estritamente ao Treino (Sem Leakage).
✓ [VALIDADO] Média das variáveis numéricas de Treino centrada em 0.0.
✓ [VALIDADO] Desvio médio entre os parâmetros do Treino vs População Total: 23910.757954


---
## 5. Teste de Robustez a Categorias Desconhecidas em Produção

Validamos a capacidade do pipeline de tolerar observações operacionais com categorias nunca antes vistas no treino (ex.: novas fontes de coleta, extensões de imagem não catalogadas ou novos bins), garantindo que o parâmetro `handle_unknown='ignore'` zere os atributos codificados sem gerar interrupções de execução (*runtime exceptions*).

In [6]:
# Criar registro de inferência contendo categorias inéditas
amostra_inedita = pd.DataFrame([{
    'mean_hue': 72.5, 'std_saturation': 0.11, 'exg_index': 48.0, 'exr_index': -12.0,
    'rg_ratio': 0.35, 'indice_clorose_necrose': 2.8, 'hue_dispersion': 9.1,
    'glcm_contrast': 11.2, 'glcm_homogeneity': 0.87, 'glcm_dissimilarity': 0.72,
    'glcm_energy': 0.89, 'indice_rugosidade_pustula': 7.9, 'laplacian_var': 135.0,
    'size_kb': 45.0, 'total_pixels': 409600, 'resolution_mp': 0.41, 'aspect_ratio': 1.0,
    # Categorias desconhecidas
    'dataset_source': 'drone_agro_coleta_2027',
    'extension': '.webp',
    'size_kb_bin': 99,
    'laplacian_var_bin': 99,
    'resolution_bin': 99,
    'aspect_ratio_bin': 99
}])

amostra_trans = preprocessor.transform(amostra_inedita)
cat_cols = [c for c in amostra_trans.columns if c.startswith('cat__')]
soma_cat = amostra_trans[cat_cols].values.sum()

print(f"• Transformação de amostra desconhecida concluída com sucesso!")
print(f"• Soma das colunas dummy geradas: {soma_cat} (Vetor nulo seguro)")
assert soma_cat == 0.0, "Categorias desconhecidas não geraram vetor nulo!"
print("✓ [VALIDADO] Resiliência a categorias inéditas confirmada.")

• Transformação de amostra desconhecida concluída com sucesso!
• Soma das colunas dummy geradas: 0.0 (Vetor nulo seguro)
✓ [VALIDADO] Resiliência a categorias inéditas confirmada.


---
## 6. Serialização MLOps com Joblib e Manifesto de Governança (Checklist 6)

Persistimos o objeto transformador ajustado em `models/preprocessor_pipeline.joblib`, validamos a integridade com hash SHA-256, testamos a idempotência numérica via `joblib.load()` e salvamos o manifesto de governança em JSON com rastreabilidade completa das features geradas.

In [7]:
models_dir = base_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)
pipeline_file = models_dir / "preprocessor_pipeline.joblib"
manifest_file = models_dir / "preprocessor_metadata.json"

# 1. Serialização
joblib.dump(preprocessor, pipeline_file, compress=3)
file_size_kb = pipeline_file.stat().st_size / 1024

# 2. Hash SHA-256
with open(pipeline_file, 'rb') as f:
    pipeline_hash = hashlib.sha256(f.read()).hexdigest()

print(f"• Artefato serializado : {pipeline_file.name} ({file_size_kb:.2f} KB)")
print(f"• SHA-256 Hash         : {pipeline_hash}")

# 3. Teste de Idempotência Numérica
preprocessor_reloaded = joblib.load(pipeline_file)
amostra_teste = df_test.head(100)

orig_out = preprocessor.transform(amostra_teste)
reloaded_out = preprocessor_reloaded.transform(amostra_teste)

np.testing.assert_allclose(orig_out.values, reloaded_out.values, rtol=1e-7, atol=1e-7)
print("✓ [VALIDADO] Idempotência matemática estrita comprovada pós-recarga.")

# 4. Inspeção das Features Finais de Saída
feature_names = preprocessor.get_feature_names_out()
print(f"\nTotal de features geradas: {len(feature_names)}")
print("Primeiras 10 features:")
for fn in feature_names[:10]:
    print(f"  - {fn}")

• Artefato serializado : preprocessor_pipeline.joblib (2.58 KB)
• SHA-256 Hash         : 95fa8356243340aa194c117b35c691871d8fc862c0db4959d512070f54c8d1f2
✓ [VALIDADO] Idempotência matemática estrita comprovada pós-recarga.

Total de features geradas: 35
Primeiras 10 features:
  - mean_hue
  - std_saturation
  - exg_index
  - exr_index
  - rg_ratio
  - indice_clorose_necrose
  - hue_dispersion
  - glcm_contrast
  - glcm_homogeneity
  - glcm_dissimilarity


---
## 7. Consolidação e Verificação da ABT de Modelagem (`abt_features_modelagem`)

Para assegurar sinergia com a próxima tarefa do sprint e preparar os dados para o `GridSearchCV` da Sprint 3, inspecionamos o arquivo consolidado `data/processed/abt_features_modelagem.parquet` gerado com os alvos preservados e 100% livre de NaNs.

In [8]:
parquet_path = base_dir / "data" / "processed" / "abt_features_modelagem.parquet"
df_final = pd.read_parquet(parquet_path)

print(f"• Dimensões da ABT Final de Modelagem : {df_final.shape[0]} linhas x {df_final.shape[1]} colunas.")
print(f"• Total de valores ausentes (NaN)     : {df_final.isna().sum().sum()}")
print(f"• Tamanho do arquivo Parquet          : {parquet_path.stat().st_size / 1024:.2f} KB")

# Amostra das primeiras colunas da ABT final
display(df_final[['sample_id', 'split_partition', 'class_label', 'target_binary'] + list(feature_names[:4])].head())

• Dimensões da ABT Final de Modelagem : 6571 linhas x 40 colunas.
• Total de valores ausentes (NaN)     : 0
• Tamanho do arquivo Parquet          : 916.92 KB


,sample_id,split_partition,class_label,target_binary,mean_hue,std_saturation,exg_index,exr_index
0,7064f596-14b4-45fc-875c-f2b0fdeceb9d,test,HEALTHY,0,1.554876,-1.711918,2.163672,-1.135561
1,4623a0fc-bec7-4ad8-a5c2-88eea86f8a0e,test,HEALTHY,0,1.381143,-0.927537,2.013610,-1.471668
2,34c5670d-c4fd-4996-bf78-83764aa6e4e3,test,HEALTHY,0,1.596528,-1.229818,1.793987,-1.635375
3,25032583-78af-426d-a4a4-5a49294a848b,test,HEALTHY,0,1.836575,-1.288724,1.518091,-1.709985
4,a252dc67-f042-4296-9d1a-0ea5d7c25224,test,HEALTHY,0,1.354836,-1.363132,0.852970,-1.808499
